# GraphMixer-Baseline – Colab-Runner

Repo `bike-link-prediction` nach Google Drive hochladen. Struktur:
```
bike-link-prediction/
├── evaluation/shared_eval.py
├── prepared Data/        (graphmixer_edges.csv, ml_citibike.* …)
└── graphmixer/model/     (graphmixer*.py, train_graphmixer.py, dieses Notebook)
```

## 1. GPU prüfen

In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Google Drive einbinden

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Pfade setzen (ROOT = Repo-Ordner in Drive anpassen)

In [ ]:
import os
ROOT = "/content/drive/MyDrive/bike-link-prediction"
MODEL_DIR = os.path.join(ROOT, "graphmixer", "model")
EVAL_DIR  = os.path.join(ROOT, "evaluation")
PREP_DIR  = os.path.join(ROOT, "prepared Data")
for p in [MODEL_DIR, EVAL_DIR, PREP_DIR]:
    print(("OK  " if os.path.isdir(p) else "FEHLT ") + p)
for f in ["graphmixer_edges.csv", "ml_citibike.csv", "ml_citibike.npy", "ml_citibike_node.npy"]:
    fp = os.path.join(PREP_DIR, f); print(("OK  " if os.path.isfile(fp) else "FEHLT ") + fp)

## 4. In den model-Ordner wechseln

In [ ]:
%cd "$MODEL_DIR" 

## 5. Konfiguration

In [ ]:
from graphmixer import GMConfig
cfg = GMConfig(epochs=20)   # GMConfig(epochs=2) für Smoke-Test
print("Epochen:", cfg.epochs, "| K:", cfg.num_neighbors, "| Mixer-Layer:", cfg.mixer_layers)

## 6. Training + Bewertung

In [ ]:
from train_graphmixer import main
main(cfg)

## 7. Ergebnisse erneut bewerten

In [ ]:
import sys, pandas as pd
sys.path.insert(0, EVAL_DIR)
from shared_eval import SharedLinkEval
ev = SharedLinkEval()
for split in ["val", "test"]:
    pred = pd.read_csv(f"predictions/graphmixer_pred_{split}.csv")
    r = ev.score_binary(pred, split=split)
    print(f"[{split}] AUC={r['auc']:.3f} AP={r['ap']:.3f} F1={r['f1']:.3f} Acc={r['accuracy']:.3f}")